<a href="https://colab.research.google.com/github/shivansh2310/Quantitative-Momentum/blob/main/The_%22Frog_in_the_Pan%22_(High_Quality_Momentum).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Quality of Momentum
Imagine two stocks that both returned +50% over the last 12 months:

* Stock A (Low Quality): Stayed completely flat for 11 months, then had a massive earnings beat and jumped 50% in a single day.

* Stock B (High Quality): Grinded upward by 0.2% every single day for 12 months with almost no volatility.

### The Behavioral Psychology:

When Stock A jumps 50% in one day, it hits the front page of the Wall Street Journal. Every retail investor notices it. The information is discrete and high-attention. This leads to an immediate overreaction, followed by long-term mean reversion (the stock will likely drop next month).

When Stock B grinds up 0.2% a day, no one notices. Like a frog in a pot of slowly boiling water, investors slowly anchor to the rising price without realizing a massive trend is underway. This information is continuous and low-attention. Because investors under-react to this slow drip of good news, the trend persists for much longer.

In Quantitative Momentum, we don't just want high returns; we want Smooth Paths.

## Path Smoothness Filter ($R^2$)

To measure this in code, quants use linear regression. We plot a perfect, straight line from the stock's price 12 months ago to its price today. Then, we calculate the $R^2$ (Coefficient of Determination) of the actual price path against that perfect line.

An $R^2$ of 1.0 means the stock went up in a perfectly straight line (Ultimate Frog-in-the-Pan). An $R^2$ of 0.1 means the stock's path was violently erratic.

In [4]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import linregress
import yfinance as yf
import warnings

warnings.filterwarnings('ignore')

In [5]:
universe = [
    'AAPL', 'MSFT', 'AMZN', 'NVDA', 'META', 'TSLA', 'GOOGL',
    'JPM', 'BAC', 'GS', 'MS', 'WFC', 'C',
    'XOM', 'CVX', 'COP', 'EOG', 'SLB',
    'JNJ', 'UNH', 'LLY', 'PFE', 'ABBV',
    'PG', 'KO', 'PEP', 'WMT', 'TGT',
    'CAT', 'DE', 'GE', 'HON', 'MMM'
]
market_proxy = 'SPY'

# Combine for downloading
download_list = universe + [market_proxy]

In [6]:

data = yf.download(download_list, period="2y")['Close']
data = data.dropna(axis=1)

[*********************100%***********************]  34 of 34 completed


In [7]:
spy_close = data[market_proxy]

In [8]:
# Calculate the 200-day Simple Moving Average (SMA)
spy_sma_200 = spy_close.rolling(window=200).mean()

In [10]:
latest_date = spy_close.index[-1]
current_spy_price = spy_close.loc[latest_date]
current_spy_sma = spy_sma_200.loc[latest_date]

is_bull_regime = current_spy_price > current_spy_sma

stock_data = data[universe] # Exclude SPY for the stock ranking

# 252 days = 12 months, 21 days = 1 month
price_t_minus_1m = stock_data.shift(21)
price_t_minus_12m = stock_data.shift(252)

academic_12m_1m_return = (price_t_minus_1m / price_t_minus_12m) - 1

# We take the top 50% of our universe based on raw momentum
current_momentum = academic_12m_1m_return.loc[latest_date].dropna()
median_momentum = current_momentum.median()
high_momentum_universe = current_momentum[current_momentum >= median_momentum].index.tolist()

In [11]:
print(f"Selected {len(high_momentum_universe)} stocks for Smoothness analysis.")

Selected 17 stocks for Smoothness analysis.


In [19]:
print("Calculating Path Smoothness (R-Squared)...")
end_date = latest_date
start_date = data.index[data.index.get_loc(end_date) - 252]
price_paths = data.loc[start_date:end_date, high_momentum_universe]

smoothness_scores = {}
X = np.arange(len(price_paths))

for ticker in high_momentum_universe:
    y = price_paths[ticker].values

    valid_idx = ~np.isnan(y)
    if valid_idx.sum() < 200:
        continue

    X_valid = X[valid_idx]
    y_valid = y[valid_idx]

    slope, intercept, r_value, p_value, std_err = linregress(X_valid, y_valid)

    r_squared = r_value ** 2
    smoothness_scores[ticker] = r_squared

Calculating Path Smoothness (R-Squared)...


In [20]:
quality_df = pd.DataFrame({
    'Raw_12m_1m': current_momentum[high_momentum_universe],
    'Smoothness_R2': pd.Series(smoothness_scores)
})

quality_df['Momentum_Rank'] = quality_df['Raw_12m_1m'].rank(ascending=True)
quality_df['Smoothness_Rank'] = quality_df['Smoothness_R2'].rank(ascending=True)

quality_df['HQ_Score'] = quality_df['Momentum_Rank'] + quality_df['Smoothness_Rank']
quality_df = quality_df.sort_values('HQ_Score', ascending=False)

print("\n" + "="*70)
print(" FINAL TARGET PORTFOLIO: HIGH-QUALITY MOMENTUM (FROG-IN-THE-PAN)")
print("="*70)
print(f"{'Ticker':<10} | {'12m-1m Return':<15} | {'Smoothness (R2)':<17} | {'HQ Score'}")
print("-" * 70)

for ticker, row in quality_df.head(5).iterrows():
    ret_str = f"{row['Raw_12m_1m']*100:.2f}%"
    r2_str = f"{row['Smoothness_R2']:.4f}"
    print(f"{ticker:<10} | {ret_str:<15} | {r2_str:<17} | {row['HQ_Score']:.1f}")
print("="*70)

top_raw_stock = quality_df['Raw_12m_1m'].idxmax()
top_hq_stock = quality_df.index[0]

if top_raw_stock != top_hq_stock:
    print(f"\n⚠️ QUALITY OVERRIDE ACTIVATED:")
    print(f"The stock with the highest raw return was {top_raw_stock}.")
    print(f"However, the system selected {top_hq_stock} as the #1 position because its price path was significantly smoother, indicating persistent behavioral under-reaction.")


 FINAL TARGET PORTFOLIO: HIGH-QUALITY MOMENTUM (FROG-IN-THE-PAN)
Ticker     | 12m-1m Return   | Smoothness (R2)   | HQ Score
----------------------------------------------------------------------
CAT        | 145.14%         | 0.9527            | 34.0
GOOGL      | 124.89%         | 0.8674            | 30.0
C          | 62.92%          | 0.8703            | 29.0
JNJ        | 55.05%          | 0.8736            | 28.0
SLB        | 63.29%          | 0.8566            | 28.0
